# Day 20：多模态输入与 Embedding 语义检索（Cosine Similarity & Semantic FAQ Matcher）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第二阶段第 20 天！今天我们将彻底打通智能体的“视觉输入协议（OpenAI Vision Content List）”与“语义理解基石（Embedding + 余弦相似度）”，用纯 Python 标准库实现确定性本地向量模型与企业级语义 FAQ 检索器（`v0.2.0-day20`）。

## 1. 环境初始化与导入 `project_iterations/day_20` 模块
本模块完全基于 Python 标准库 `math` 与 `hashlib` 构建，无需外部 `numpy` 或在线 Embedding API 即可离线确定性运行。

In [1]:
import json
import math
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day20_dir = repo_root / "project_iterations" / "day_20"
if str(day20_dir) not in sys.path:
    sys.path.insert(0, str(day20_dir))

from zhishu_agent import (
    LocalEmbeddingModel,
    SemanticFAQMatcher,
    build_multimodal_message,
    cosine_similarity,
    create_default_faq_matcher,
)

print("Day 20 模块导入成功:", day20_dir)

Day 20 模块导入成功: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_20


## 2. 构造标准 OpenAI Vision 多模态消息
在多模态对话中，`user` 消息的 `content` 不再是单一字符串，而是由 `text` 与 `image_url` 对象组成的列表。

In [2]:
mm_message = build_multimodal_message(
    text="请核对这张差旅电子发票上的金额与开票日期。",
    image_url="https://example.xingchen.local/assets/invoice_2026.png",
    detail="high",
)
print(json.dumps(mm_message, ensure_ascii=False, indent=2))

{
  "role": "user",
  "content": [
    {
      "type": "text",
      "text": "请核对这张差旅电子发票上的金额与开票日期。"
    },
    {
      "type": "image_url",
      "image_url": {
        "url": "https://example.xingchen.local/assets/invoice_2026.png",
        "detail": "high"
      }
    }
  ]
}


## 3. 确定性 Embedding 与余弦相似度数学验证
验证 `LocalEmbeddingModel` 输出单位向量（L2 范数 = 1.0），并对比同义句与无关句的余弦相似度差异。

In [3]:
model = LocalEmbeddingModel(dimensions=128)
v_base = model.embed("差旅费用和发票怎么提交报销？")
v_sim = model.embed("出差回来发票去哪里报销")
v_diff = model.embed("星辰大厦三楼会议室怎么预订")

norm = math.sqrt(sum(x * x for x in v_base))
print(f"向量维度: {len(v_base)} | L2 范数: {norm:.6f}")
print(f"同义句余弦相似度: {cosine_similarity(v_base, v_sim):.4f}")
print(f"无关句余弦相似度: {cosine_similarity(v_base, v_diff):.4f}")
print(f"正交向量 [1,0] vs [0,1]: {cosine_similarity([1.0, 0.0], [0.0, 1.0]):.4f}")

向量维度: 128 | L2 范数: 1.000000
同义句余弦相似度: 0.3492
无关句余弦相似度: 0.2303
正交向量 [1,0] vs [0,1]: 0.0000


## 4. 企业语义 FAQ 检索器（`SemanticFAQMatcher`）实战
加载星辰数科默认企业知识库，测试口语化自然语言提问的 Top-2 语义召回效果。

In [4]:
matcher = create_default_faq_matcher(dimensions=128)
queries = [
    "出差发票怎么报销？",
    "VPN 密码忘记了怎么重置？",
    "我想休年假怎么申请？",
]

for q in queries:
    print(f"\n[提问] {q}")
    for idx, match in enumerate(matcher.search(q, top_k=2), start=1):
        print(f"  Top-{idx} (score={match.score:.4f}) [{match.category}] {match.question}")
        print(f"         答案: {match.answer}")


[提问] 出差发票怎么报销？
  Top-1 (score=0.5838) [财务制度] 差旅费用和发票怎么提交报销？
         答案: 请登录智枢 OA 系统 -> 财务报销 -> 差旅报销，上传电子发票 PDF 并关联出差审批单，每周五统一打款。
  Top-2 (score=0.1313) [人事考勤] 员工年假怎么申请以及有多少天？
         答案: 司龄满 1 年享 5 天带薪年假，满 5 年享 10 天；在 OA 请假模块提交申请并由直属主管审批即可。

[提问] VPN 密码忘记了怎么重置？
  Top-1 (score=0.6824) [IT 支持] 公司内网 VPN 密码忘记了如何重置？
         答案: 访问 IT 自助服务台（it.xingchen.local）点击“忘记密码”，通过企业微信扫码验证后即可重置 VPN 密码。
  Top-2 (score=0.1418) [人事考勤] 员工年假怎么申请以及有多少天？
         答案: 司龄满 1 年享 5 天带薪年假，满 5 年享 10 天；在 OA 请假模块提交申请并由直属主管审批即可。

[提问] 我想休年假怎么申请？
  Top-1 (score=0.5538) [人事考勤] 员工年假怎么申请以及有多少天？
         答案: 司龄满 1 年享 5 天带薪年假，满 5 年享 10 天；在 OA 请假模块提交申请并由直属主管审批即可。
  Top-2 (score=0.0523) [IT 支持] 公司内网 VPN 密码忘记了如何重置？
         答案: 访问 IT 自助服务台（it.xingchen.local）点击“忘记密码”，通过企业微信扫码验证后即可重置 VPN 密码。
